# Spending Generalization Research V3
Standalone, CPU-bounded synthetic experiments, not app integration. Run all with numpy, pandas, scikit-learn, matplotlib and joblib already installed (standard Colab). No repository imports, external data, automatic pip upgrades, uploads, or downloads.

**Protocol:** Development seeds and final seeds below are precommitted. Repeated development is allowed. Select forecasting models on two temporal development folds, and anomaly models with an earlier threshold slice followed by later validation in each fold. Refit on development only, then freeze anomaly thresholds on a separate later calibration period. Evaluate known-user future and disjoint development users without further selection. Final is OFF by default: freeze first, explicitly opt in, evaluate ALL declared seeds and both processes exactly once. Once viewed, final seeds are consumed, even if you change output directories. The local marker is a convenience guard, not a registry.

**Why a new benchmark:** V2's final-seed generator belonged to the development family. Its approximately 30% WAPE and .81 precision/recall are not comparable here. This benchmark adds a different shifted process, benign lookalikes, and final-only gradual/split anomalies with lower shifted-final prevalence. The Sunday recurrence interval is half-open through next Monday, not truncated at Sunday midnight. Linear/tabular candidates are sufficient for this small CPU experiment; no LSTM.

High precision, recall, and generalization are objectives, not promises. Synthetic labels encode generator assumptions; unusual legitimate purchases can look identical to injected events. Neither this benchmark nor its confidence intervals establish real-world accuracy. Real consented data, independent labels, and a common untouched test set are needed before any historical leaderboard or deployment claim.


In [ ]:
import os, sys, json, time, hashlib, importlib, importlib.metadata, marshal, io
from pathlib import Path
from collections import defaultdict, deque
from datetime import datetime, timezone
# Set these before importing numerical libraries; threadpoolctl also limits loaded BLAS.
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(key, '1')
PACKAGES = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn',
            'matplotlib': 'matplotlib', 'joblib': 'joblib'}
missing = [name for name in PACKAGES if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Install missing dependencies in a fresh runtime, then restart: ' + str(missing))
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest
from sklearn.metrics import (precision_recall_curve, average_precision_score, roc_auc_score, r2_score)
from threadpoolctl import threadpool_limits
THREAD_LIMIT = threadpool_limits(limits=1)
MODE = os.environ.get('SPENDLY_MODE', 'quick').lower()
MODES = {'smoke': dict(users=16, weeks=24, trees=40, iterations=45, bootstrap=50),
         'quick': dict(users=60, weeks=40, trees=80, iterations=90, bootstrap=200),
         'full': dict(users=150, weeks=52, trees=120, iterations=140, bootstrap=300)}
if MODE not in MODES:
    raise ValueError('SPENDLY_MODE must be smoke, quick, or full')
CFG = MODES[MODE]
DEV_SEEDS, FINAL_SEEDS = (1207, 3911), (7013, 9029)
ENVIRONMENTS = ('baseline', 'shifted')
RUN_FINAL = os.environ.get('SPENDLY_RUN_FINAL', '0') == '1'
MIN_PRECISION = float(os.environ.get('SPENDLY_MIN_PRECISION', '0.80'))
MAX_FPR = float(os.environ.get('SPENDLY_MAX_FPR', '0.02'))
assert 0 <= MIN_PRECISION <= 1 and 0 <= MAX_FPR <= 1
OUTPUT_DIR = Path(os.environ.get('SPENDLY_OUTPUT_DIR', 'research_v3_outputs'))
if (OUTPUT_DIR / 'final_consumed.json').exists():
    raise RuntimeError('Final seeds already consumed here; preserve these artifacts. Do not reuse viewed seeds for final claims.')
FORECAST_CANDIDATES = ('last', 'mean4', 'recurring', 'ridge_ratio', 'hgb_ratio', 'hgb_raw')
ANOMALY_CANDIDATES = ('rules', 'iforest', 'hybrid')
VERSIONS = {dist: importlib.metadata.version(dist) for dist in PACKAGES.values()}
VERSIONS.update(python=sys.version, threadpoolctl=importlib.metadata.version('threadpoolctl'))
STARTED = time.perf_counter()
print(json.dumps(dict(mode=MODE, config=CFG, final=RUN_FINAL, versions=VERSIONS), indent=2))


## Observable Data And Ground Truth
Each profile declares complete observation coverage [start, end), including zero-spend days, and a static monthly income known at observation start. All amounts are KES; median generated monthly income is KES 34,000. All transactions are positive expenses. Labels and event IDs live in a separate truth table and are joined only for training selection/evaluation, never feature construction. IDs include seed and environment. Per-user normal and injection RNG streams are independent.
Baseline uses fairly stable category preferences and light-tailed daily spending; shifted uses a persistent weekly spending state, different category/weekday preferences, heavier tails, merchant churn, and shifted recurring amounts/dates. Both include legitimate repeats, night activity, bursts, and planned large purchases. Final shifted data alone includes gradual/split behavior and fewer injected events. This is a controlled synthetic shift, not a substitute for unknown real distributions.


In [ ]:
def generate(seed, environment, users=None, weeks=None, final=False, inject=True):
    users, weeks = users or CFG['users'], weeks or CFG['weeks']
    shifted = environment == 'shifted'
    start = pd.Timestamp('2025-01-06' if final else '2023-01-02')
    end = start + pd.Timedelta(weeks=weeks)
    rows, profiles, truth = [], [], []
    for u in range(users):
        uid = f'{seed}_{environment}_u{u:03d}'
        normal = np.random.default_rng(np.random.SeedSequence([seed, int(shifted), u, 11]))
        anomaly = np.random.default_rng(np.random.SeedSequence([seed, int(shifted), u, 29]))
        income = float(normal.lognormal(np.log(34000), .45))
        profiles.append(dict(user=uid, seed=seed, environment=environment, income=income,
                             start=start, end=end, cohort='unseen' if u % 5 == 0 else 'known'))
        local, annotations = [], {}
        def add(t, amount, category, merchant, suffix=None, event=None, family=None):
            tid = f'{uid}_{suffix or ("n" + str(len(local)))}'
            local.append(dict(transaction=tid, user=uid, timestamp=pd.Timestamp(t),
                              amount=round(max(.50, float(amount)), 2), category=category, merchant=merchant))
            if event is not None:
                annotations[tid] = (event, family)
        preference = normal.dirichlet([6, 4, 3, 2] if not shifted else [3, 3, 5, 4])
        state = 0.0
        for w in range(weeks):
            state = .72 * state + normal.normal(0, .26) if shifted else normal.normal(0, .07)
            n = normal.poisson(9 if not shifted else 11)
            for j in range(n):
                day = int(normal.choice(7, p=[.12, .12, .12, .12, .16, .20, .16] if shifted else None))
                hour = int(normal.choice([1, 8, 12, 17, 20, 23], p=[.04, .12, .29, .24, .24, .07]))
                t = start + pd.Timedelta(days=w * 7 + day, hours=hour, minutes=int(normal.integers(60)))
                c = int(normal.choice(4, p=preference))
                category = ('food', 'transport', 'shopping', 'leisure')[c]
                merchant = f'{category}_{normal.integers(3 + (3 if shifted and w > weeks // 2 else 0))}'
                amount = income * (.009, .006, .016, .012)[c] * np.exp(state + normal.normal(0, .75 if shifted else .40))
                add(t, amount, category, merchant)
                if normal.random() < .025:  # A benign repeat must not automatically mean fraud.
                    add(t + pd.Timedelta(minutes=4), amount, category, merchant)
            if normal.random() < .13:
                t = start + pd.Timedelta(weeks=w, days=5, hours=22)
                for j in range(int(normal.integers(3, 6))):
                    add(t + pd.Timedelta(minutes=j * 9), income * normal.uniform(.003, .012), 'leisure', f'outing_{j % 2}')
        months = pd.date_range(start - pd.offsets.MonthBegin(1), end, freq='MS')
        for category, merchant, fraction, day in [('housing', 'rent', .25, 7), ('services', 'phone', .015, 15),
                                                   ('services', 'streaming', .006, 24)]:
            for k, month in enumerate(months):
                t = month + pd.Timedelta(days=day - 1 + (2 if shifted and k > 4 else 0), hours=18)
                if start <= t < end:
                    add(t, income * fraction * (1.10 if shifted and k > 4 else 1), category, merchant)
        for j in range(int(normal.integers(1, 4))):
            t = start + pd.Timedelta(days=int(normal.integers(weeks * 7)), hours=14)
            add(t, income * normal.uniform(.12, .45), 'shopping', 'planned_store')
        # Only the independent anomaly stream is used below this point.
        families = ['large', 'burst_duplicate', 'recurring_change']
        if final and shifted:
            families += ['gradual', 'split']
        count = anomaly.poisson(1.2 if final and shifted else 2.4) if inject else 0
        changed = set()
        for e in range(count):
            family = str(anomaly.choice(families))
            event = f'{uid}_event{e}'
            t = start + pd.Timedelta(weeks=int(anomaly.integers(8, weeks)),
                                     days=int(anomaly.integers(7)), hours=int(anomaly.choice([1, 13, 21])))
            if family == 'recurring_change':
                merchant = str(anomaly.choice(['phone', 'streaming', 'rent']))
                affected = sorted([r for r in local if r['merchant'] == merchant and r['timestamp'] >= t
                                   and r['transaction'] not in changed], key=lambda r: r['timestamp'])[:2]
                for row in affected:
                    row['amount'] = round(row['amount'] * anomaly.uniform(1.8, 3.0), 2)
                    annotations[row['transaction']] = (event, family)
                    changed.add(row['transaction'])
                continue
            repetitions = {'large': 1, 'burst_duplicate': int(anomaly.integers(3, 7)), 'gradual': 3, 'split': 7}[family]
            base = income * anomaly.uniform(.13, .35)
            for j in range(repetitions):
                delta = pd.Timedelta(weeks=j) if family == 'gradual' else pd.Timedelta(days=j) if family == 'split' else pd.Timedelta(minutes=j * 2)
                amount = base * (1 + .25 * j) if family == 'gradual' else base / 5 if family == 'split' else base
                merchant = f'shopping_{j % 3}' if family == 'split' else 'shopping_0'
                if t + delta < end:
                    add(t + delta, amount, 'shopping', merchant, f'a{e}_{j}', event, family)
        rows.extend(local)
        for row in local:
            event, family = annotations.get(row['transaction'], (None, None))
            truth.append(dict(transaction=row['transaction'], label=int(event is not None), event=event, family=family))
    tx = pd.DataFrame(rows).sort_values(['user', 'timestamp', 'transaction']).reset_index(drop=True)
    return dict(tx=tx, profiles=pd.DataFrame(profiles), truth=pd.DataFrame(truth))

def complete_mondays(start, end):
    first = start.normalize() - pd.Timedelta(days=start.weekday())
    if first < start:
        first += pd.Timedelta(weeks=1)
    last = end.normalize() - pd.Timedelta(days=end.weekday())
    return pd.date_range(first, last, freq='7D')

def recurrence(history, left, right, window_start=None):
    history = history.loc[history.timestamp.lt(left)]
    predicted, recurring_past = 0.0, 0.0
    for _, group in history.groupby(['merchant', 'category'], sort=False):
        group = group.sort_values('timestamp')
        dates = group.timestamp.drop_duplicates()
        if len(dates) < 2:
            continue
        gaps = np.diff(dates.to_numpy()) / np.timedelta64(1, 'D')
        cadence = float(np.median(gaps))
        if not (5 <= cadence <= 40) or np.median(np.abs(gaps - cadence)) > .22 * cadence:
            continue
        # Monthly date arithmetic, not 30-day drift. Forecast interval includes all Sunday.
        step = pd.DateOffset(months=1) if 26 <= cadence <= 33 else pd.Timedelta(days=round(cadence))
        due, amount = dates.iloc[-1] + step, float(group.amount.tail(3).median())
        if due < left - pd.Timedelta(days=40):
            continue
        while due < right:
            if left <= due < right:
                predicted += amount
            due += step
        recurring_past += float(group.loc[group.timestamp.ge(window_start), 'amount'].sum() if window_start is not None else group.amount.sum())
    return predicted, recurring_past

F_FEATURES = ([f'lag{i}_ratio' for i in range(1, 9)] +
              ['count1', 'count4_mean', 'mean_amount_ratio', 'std8_ratio', 'recurring_ratio',
               'nonrecurring_ratio', 'month_sin', 'month_cos', 'income_to_scale', 'zero_week_fraction', 'log_past_scale_KES'])

def build_forecast(tx, profiles):
    records, features = [], []
    for p in profiles.itertuples(index=False):
        data = tx.loc[(tx.user == p.user) & (tx.timestamp >= p.start) & (tx.timestamp < p.end)]
        boundaries = complete_mondays(p.start, p.end)
        if len(boundaries) < 10:
            continue
        weekly = data.assign(week=data.timestamp.dt.normalize() - pd.to_timedelta(data.timestamp.dt.weekday, unit='D'))
        grouped = weekly.groupby('week').amount.agg(['sum', 'count']).reindex(boundaries[:-1], fill_value=0)
        amounts, counts = grouped['sum'].to_numpy(), grouped['count'].to_numpy()
        for i in range(8, len(boundaries) - 1):
            left, right = boundaries[i], boundaries[i + 1]
            past = amounts[i - 8:i]
            scale = max(float(np.median(past)), p.income / 4.345 * .10, 1.0)
            hist = data.loc[(data.timestamp >= left - pd.Timedelta(weeks=26)) & (data.timestamp < left)]
            recurring, rec_past = recurrence(hist, left, right, window_start=boundaries[i - 8])
            nonrec = max(0., (past.sum() - rec_past) / 8)
            x = list(past[::-1] / scale) + [counts[i - 1], counts[i - 4:i].mean(),
                past.sum() / max(1, counts[i - 8:i].sum()) / scale, past.std() / scale,
                recurring / scale, nonrec / scale, np.sin(2 * np.pi * left.month / 12),
                np.cos(2 * np.pi * left.month / 12), p.income / scale, np.mean(past == 0), np.log(scale)]
            features.append(x)
            records.append(dict(user=p.user, seed=p.seed, environment=p.environment, cohort=p.cohort,
                                time=left, target_end=right, week=(left - p.start).days // 7,
                                y=amounts[i], scale=scale, last=past[-1], mean4=past[-4:].mean(), recurring=recurring + nonrec))
    return pd.DataFrame(records), pd.DataFrame(features, columns=F_FEATURES, dtype=float)

A_FEATURES = ['amount_income', 'amount_ratio', 'category_deviation', 'merchant_ratio',
              'count_hour', 'sum_hour_ratio', 'count_day', 'duplicate_hour', 'night_rarity',
              'category_cold', 'merchant_cold', 'user_cold', 'hour_sin', 'hour_cos']

def build_anomaly(tx, profiles):
    records, features = [], []
    for p in profiles.itertuples(index=False):
        data = tx.loc[(tx.user == p.user) & (tx.timestamp >= p.start) & (tx.timestamp < p.end)]
        data = data.sort_values(['timestamp', 'transaction'])
        amounts, categories, merchants, recent = deque(maxlen=80), defaultdict(lambda: deque(maxlen=40)), defaultdict(lambda: deque(maxlen=12)), deque()
        nights, seen = 0, 0
        for r in data.itertuples(index=False):
            t, a = r.timestamp, r.amount
            while recent and recent[0][0] < t - pd.Timedelta(days=1):
                recent.popleft()
            hour = [q for q in recent if q[0] >= t - pd.Timedelta(hours=1)]
            cat, mer = categories[r.category], merchants[r.merchant]
            typical = max(float(np.median(amounts)) if amounts else p.income * .01, 1.)
            center = float(np.median(cat)) if cat else typical
            spread = max(float(np.median(np.abs(np.asarray(cat) - center))) * 1.4826 if cat else 0., center * .25, 1.)
            merchant_center = max(float(np.median(mer)) if mer else typical, 1.)
            night = int(t.hour < 5 or t.hour >= 23)
            # All history features are computed BEFORE incorporating the current transaction.
            features.append([a / p.income, min(a / typical, 100), np.clip((a - center) / spread, -100, 100),
                min(a / merchant_center, 100), len(hour), sum(q[1] for q in hour) / typical, len(recent),
                sum(q[2] == r.merchant and abs(q[1] - a) < .01 for q in hour),
                night * (1 - (nights + 1) / (seen + 2)), len(cat) < 3, len(mer) < 2, seen < 10,
                np.sin(2 * np.pi * t.hour / 24), np.cos(2 * np.pi * t.hour / 24)])
            records.append(dict(transaction=r.transaction, user=p.user, seed=p.seed, environment=p.environment,
                                cohort=p.cohort, time=t, week=(t - p.start).days // 7))
            amounts.append(a); cat.append(a); mer.append(a)
            recent.append((t, a, r.merchant)); nights += night; seen += 1
    return pd.DataFrame(records), pd.DataFrame(features, columns=A_FEATURES, dtype=float)

def prepare(seeds, final=False):
    bundles = [generate(seed, env, final=final) for seed in seeds for env in ENVIRONMENTS]
    profiles = pd.concat([b['profiles'] for b in bundles], ignore_index=True)
    tx = pd.concat([b['tx'] for b in bundles], ignore_index=True)
    truth = pd.concat([b['truth'] for b in bundles], ignore_index=True)
    f, fx = build_forecast(tx, profiles)
    a, ax = build_anomaly(tx, profiles)
    a = a.merge(truth, on='transaction', how='left', validate='one_to_one', sort=False)
    assert len(a) == len(ax) and a.label.notna().all()
    assert np.isfinite(fx.to_numpy()).all() and np.isfinite(ax.to_numpy()).all()
    assert not set(F_FEATURES + A_FEATURES) & {'label', 'family', 'event', 'transaction', 'user', 'seed', 'environment'}
    return dict(f=f, fx=fx, a=a, ax=ax, profiles=profiles)


## Metrics And Constraints
Forecasts use eight completed weeks to predict the next completed Monday-to-Monday week. All models receive log_past_scale_KES as well as normalized lags, making raw/ratio target ablation fair. Recurrence inference uses up to 26 past weeks; only recurring amounts inside the eight-week lag window are subtracted from that window's total. No user embeddings or fitting on unseen-user warmup.
Event purging changes selection masks only, never the full causal feature streams. Any positive event not wholly contained in one fit/calibration/validation mask is excluded from all those masks, including events beginning in an unselected gap. Freeze masks also purge across the later evaluation boundary. Transaction evaluation intentionally retains continuations; new-event metrics left-censor them. Every anomaly candidate gets its own threshold on the SAME later calibration rows, and every candidate is evaluated on identical rows without final reselection.
Anomaly scores may use the current transaction, never future history. Rules and Isolation Forest are unsupervised fits; development labels select models/thresholds, so the overall workflow is label-informed. No supervised classifier is added. IF reference CDFs and hybrid components are fit on training only and exported. Benign/anomalous training contamination is intentional.
Precision and recall compete, especially at low prevalence. Threshold constraints are empirical calibration point estimates, not guarantees or confidence bounds. No feasible threshold means **constraints unmet**, with a predeclared finite no-alert threshold above 1. No-alert precision/recall/F1 are reported as zero, not successful precision. AP/AUC are undefined (NaN) when a required class is absent. Event recall counts an event if any affected transaction is alerted; delay starts at its first affected transaction. Events already underway before an evaluation slice are reported as left-censored and excluded from event recall/delay, not silently treated as new events.
User-cluster bootstrap intervals resample entire users within each seed/environment evaluation group, conditional on the frozen model. They do not measure training uncertainty, population validity, or seed-level uncertainty. Seed summaries retain every seed, with no best-seed selection.


In [ ]:
def forecast_metrics(y, prediction):
    y, prediction = np.asarray(y, float), np.asarray(prediction, float)
    error = prediction - y
    return dict(n=len(y), MAE=float(np.abs(error).mean()), RMSE=float(np.sqrt(np.mean(error ** 2))),
                WAPE=float(np.abs(error).sum() / y.sum()) if y.sum() else np.nan,
                R2=float(r2_score(y, prediction)) if len(y) > 1 and np.var(y) > 0 else np.nan, bias=float(error.mean()))

def anomaly_metrics(y, scores, alerts):
    y, scores, alerts = np.asarray(y, bool), np.asarray(scores, float), np.asarray(alerts, bool)
    tp, fp, tn, fn = int((y & alerts).sum()), int((~y & alerts).sum()), int((~y & ~alerts).sum()), int((y & ~alerts).sum())
    precision, recall = tp / max(1, tp + fp), tp / max(1, tp + fn)
    both = y.any() and (~y).any()
    return dict(n=len(y), positive_support=int(y.sum()), negative_support=int((~y).sum()),
                precision=precision, recall=recall, F1=2 * precision * recall / max(1e-15, precision + recall),
                accuracy=(tp + tn) / max(1, len(y)), balanced_accuracy=(recall + tn / (tn + fp)) / 2 if both else np.nan,
                AP=float(average_precision_score(y, scores)) if both else np.nan,
                ROC_AUC=float(roc_auc_score(y, scores)) if both else np.nan,
                FPR=fp / (fp + tn) if fp + tn else np.nan, TP=tp, FP=fp, TN=tn, FN=fn, alert_rate=float(alerts.mean()) if len(y) else 0.)

def choose_threshold(y, scores):
    y, scores = np.asarray(y, int), np.asarray(scores, float)
    fallback = dict(threshold=float(np.nextafter(1., np.inf)), feasible=False, reason='constraints unmet; no-alert fallback')
    assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()
    if not len(y) or len(np.unique(y)) < 2:
        return {**fallback, 'reason': 'calibration requires both classes; no-alert fallback'}
    precision, recall, thresholds = precision_recall_curve(y, scores)
    # The final precision/recall point has NO corresponding threshold.
    precision, recall = precision[:-1], recall[:-1]
    sorted_scores = np.sort(scores)
    alerts = len(scores) - np.searchsorted(sorted_scores, thresholds, side='left')
    tp = np.rint(recall * y.sum())
    fpr = (alerts - tp) / (len(y) - y.sum())
    feasible = np.flatnonzero((precision >= MIN_PRECISION) & (fpr <= MAX_FPR + 1e-12) & (alerts > 0))
    if not len(feasible):
        return fallback
    best = max(feasible, key=lambda i: (recall[i], precision[i], thresholds[i]))
    return dict(threshold=float(thresholds[best]), feasible=True, reason='calibration point constraints met')

def fit_forecast(name, x, frame):
    if name in ('last', 'mean4', 'recurring'):
        return None
    model = (make_pipeline(StandardScaler(), Ridge(alpha=30.)) if name == 'ridge_ratio' else
             HistGradientBoostingRegressor(loss='absolute_error', max_iter=CFG['iterations'], max_leaf_nodes=15,
                                           min_samples_leaf=30, l2_regularization=10., early_stopping=False, random_state=47))
    target = frame.y if name == 'hgb_raw' else frame.y / frame.scale
    return model.fit(x, target)

def predict_forecast(name, model, x, frame):
    if model is None:
        return frame[name].to_numpy()
    prediction = model.predict(x)
    return np.maximum(0., prediction if name == 'hgb_raw' else prediction * frame.scale.to_numpy())

def rule_scores(x):
    return (np.log1p(np.maximum(0, x.category_deviation.to_numpy())) +
            .6 * np.log1p(x.amount_ratio.to_numpy()) + .7 * np.log1p(x.merchant_ratio.to_numpy()) +
            .9 * np.log1p(x.duplicate_hour.to_numpy()) + .3 * np.log1p(x.count_hour.to_numpy()) +
            .2 * x.night_rarity.to_numpy())

def fit_anomaly(x):
    rng = np.random.default_rng(47)
    reference = x.iloc[np.sort(rng.choice(len(x), min(len(x), 20000), replace=False))]
    model = IsolationForest(n_estimators=CFG['trees'], max_samples=min(512, len(reference)),
                            contamination='auto', random_state=47, n_jobs=1).fit(reference)
    return dict(model=model, rule_cdf=np.sort(rule_scores(reference)),
                if_cdf=np.sort(-model.score_samples(reference)))

def score_anomaly(fitted, x):
    def cdf(reference, values):
        return np.searchsorted(reference, values, side='right') / len(reference)
    rules = cdf(fitted['rule_cdf'], rule_scores(x))
    forest = cdf(fitted['if_cdf'], -fitted['model'].score_samples(x))
    return dict(rules=rules, iforest=forest, hybrid=.65 * rules + .35 * forest)

def split_masks(frame, fold, forecast=False):
    weeks = CFG['weeks']
    train_end = int(weeks * (.40 if fold == 0 else .52))
    cal_end, val_end = train_end + max(1, int(.08 * weeks)), train_end + max(1, int(.08 * weeks)) + max(2, int(.10 * weeks))
    known = frame.cohort.eq('known')
    available = frame.week + (1 if forecast else 0)
    train = known & (available <= train_end if forecast else frame.week < train_end)
    calibration = known & frame.week.ge(train_end) & frame.week.lt(cal_end)
    validation = known & frame.week.ge(cal_end) & frame.week.lt(val_end)
    assert not (train & calibration).any() and not (train & validation).any() and not (calibration & validation).any()
    if forecast:
        assert frame.loc[train, 'target_end'].max() <= frame.loc[validation, 'time'].min()
    else:
        train, calibration, validation = purge_event_masks(frame, [train, calibration, validation])
        assert frame.loc[train, 'time'].max() < frame.loc[calibration, 'time'].min()
        assert frame.loc[calibration, 'time'].max() < frame.loc[validation, 'time'].min()
    assert set(frame.loc[train | calibration | validation, 'user']).isdisjoint(frame.loc[~known, 'user'])
    return train, calibration, validation

def purge_event_masks(frame, masks):
    # Full-event containment also removes gap onsets and continuations past a mask's end.
    positive = frame.label.eq(1) & frame.event.notna()
    total = frame.loc[positive].groupby('event').size()
    clean = []
    for mask in masks:
        count = frame.loc[positive & mask].groupby('event').size()
        contained = count.index[count.eq(total.reindex(count.index))]
        clean.append(mask & (~positive | frame.event.isin(contained)))
    sets = [set(frame.loc[m & positive, 'event']) for m in clean]
    assert all(sets[i].isdisjoint(sets[j]) for i in range(len(sets)) for j in range(i))
    return clean


In [ ]:
def run_correctness_tests():
    fixture = generate(83, 'baseline', users=2, weeks=24)
    tx, profiles = fixture['tx'], fixture['profiles']
    f, fx = build_forecast(tx, profiles)
    a, ax = build_anomaly(tx, profiles)
    cut = profiles.start.iloc[0] + pd.Timedelta(weeks=16)
    changed = tx.copy()
    changed.loc[changed.timestamp >= cut, 'amount'] *= 30
    changed.loc[changed.timestamp >= cut, 'category'] = 'future_mutation'
    f2, fx2 = build_forecast(changed, profiles)
    a2, ax2 = build_anomaly(changed, profiles)
    np.testing.assert_array_equal(fx.loc[f.time <= cut], fx2.loc[f2.time <= cut])
    np.testing.assert_array_equal(ax.loc[a.time < cut], ax2.loc[a2.time < cut])
    poisoned = tx.assign(label=1, event='forbidden', family='forbidden')
    np.testing.assert_array_equal(fx, build_forecast(poisoned, profiles)[1])
    np.testing.assert_array_equal(ax, build_anomaly(poisoned, profiles)[1])
    assert len(fixture['truth']) == len(tx) and tx.amount.gt(0).all()
    normal_only = generate(83, 'baseline', users=2, weeks=24, inject=False)['tx'].set_index('transaction')
    benign_ids = fixture['truth'].loc[fixture['truth'].label.eq(0), 'transaction']
    pd.testing.assert_frame_equal(tx.set_index('transaction').loc[benign_ids], normal_only.loc[benign_ids])
    pd.testing.assert_frame_equal(profiles, generate(83, 'baseline', users=2, weeks=24, inject=False)['profiles'])
    zero_f, zero_x = build_forecast(tx.iloc[:0], profiles)
    assert len(zero_f) == len(f) and zero_f.y.eq(0).all() and np.isfinite(zero_x.to_numpy()).all()
    assert not fixture['truth'].family.isin(['gradual', 'split']).any()
    left, right = pd.Timestamp('2023-04-03'), pd.Timestamp('2023-04-10')
    sunday = pd.DataFrame(dict(merchant=['m'] * 2, category=['c'] * 2, amount=[10., 10.],
                               timestamp=pd.to_datetime(['2023-02-09 23:00', '2023-03-09 23:00'])))
    monday = sunday.assign(timestamp=pd.to_datetime(['2023-02-10', '2023-03-10']))
    assert recurrence(sunday, left, right)[0] == 10., 'Sunday evening is inside the target week'
    assert recurrence(monday, left, right)[0] == 0., 'Next Monday midnight is outside'
    monthly = pd.DataFrame(dict(merchant=['m'] * 3, category=['c'] * 3, amount=[10.] * 3,
                                timestamp=pd.to_datetime(['2023-01-10', '2023-02-10', '2023-03-10'])))
    due = pd.Timestamp('2023-04-10')
    assert recurrence(monthly, due, due + pd.Timedelta(weeks=1), due - pd.Timedelta(weeks=8)) == (10., 10.)
    np.testing.assert_allclose(np.exp(fx.log_past_scale_KES), f.scale)
    boundary = pd.DataFrame(dict(event=['cross', 'cross', 'gap', 'gap', 'train', 'cal', 'val', 'late', 'late'],
                                 label=[1] * 9, week=[1, 4, 3, 4, 1, 4, 6, 6, 8]))
    masks = [boundary.week.lt(2), boundary.week.between(4, 5), boundary.week.between(6, 7)]
    cleaned = purge_event_masks(boundary, masks)
    assert [set(boundary.loc[m, 'event']) for m in cleaned] == [{'train'}, {'cal'}, {'val'}]
    onsets = boundary.groupby('event').week.min()
    assert boundary.loc[cleaned[1], 'event'].map(onsets).ge(4).all()
    assert len(boundary) == 9, 'Purging must not delete feature-stream rows'
    partial = profiles.copy()
    partial['start'] += pd.Timedelta(days=1)
    partial['end'] -= pd.Timedelta(days=2)
    pf, _ = build_forecast(tx, partial)
    assert pf.time.min() >= profiles.start.iloc[0] + pd.Timedelta(weeks=9)
    assert pf.target_end.max() <= profiles.end.iloc[0] - pd.Timedelta(weeks=1)
    assert (f.target_end - f.time).eq(pd.Timedelta(weeks=1)).all()
    empty = anomaly_metrics([0, 0], [.1, .2], [False, False])
    assert empty['precision'] == 0 and empty['recall'] == 0 and np.isnan(empty['ROC_AUC']) and np.isnan(empty['AP'])
    assert not choose_threshold([0, 0], [.1, .2])['feasible']
    # Check PR threshold alignment and fallback semantics by recomputing actual metrics.
    chosen = choose_threshold([0, 0, 1, 1], [.1, .2, .8, .9])
    measured = anomaly_metrics([0, 0, 1, 1], [.1, .2, .8, .9], np.array([.1, .2, .8, .9]) >= chosen['threshold'])
    assert chosen['feasible'] and measured['precision'] >= MIN_PRECISION and measured['FPR'] <= MAX_FPR
    assert np.isfinite(fx.to_numpy()).all() and np.isfinite(ax.to_numpy()).all()
    print('Correctness tests passed: future invariance, label blindness, independent RNG, Sunday, coverage/zero weeks, absent classes, thresholds.')

run_correctness_tests()
development = prepare(DEV_SEEDS)
for fold in (0, 1):
    split_masks(development['f'], fold, forecast=True)
    split_masks(development['a'], fold)
print('Chronology and unseen-user disjointness passed before training.')
print('Development rows:', {key: len(development[key]) for key in ('f', 'a', 'profiles')})


In [ ]:
def select_models(data):
    f, fx, a, ax = (data[k] for k in ('f', 'fx', 'a', 'ax'))
    forecast_rows, anomaly_rows, threshold_rows = [], [], []
    for fold in (0, 1):
        train, _, validation = split_masks(f, fold, forecast=True)
        for name in FORECAST_CANDIDATES:
            fitted = fit_forecast(name, fx.loc[train], f.loc[train])
            evaluated = f.loc[validation].copy()
            evaluated['prediction'] = predict_forecast(name, fitted, fx.loc[validation], evaluated)
            for (seed, env), group in evaluated.groupby(['seed', 'environment']):
                forecast_rows.append(dict(fold=fold, seed=seed, environment=env, model=name, **forecast_metrics(group.y, group.prediction)))
        train, calibration, validation = split_masks(a, fold)
        fitted = fit_anomaly(ax.loc[train])
        cal_scores, val_scores = score_anomaly(fitted, ax.loc[calibration]), score_anomaly(fitted, ax.loc[validation])
        for name in ANOMALY_CANDIDATES:
            threshold = choose_threshold(a.loc[calibration, 'label'], cal_scores[name])
            threshold_rows.append(dict(fold=fold, model=name, **threshold))
            evaluated = a.loc[validation].copy()
            evaluated['score'], evaluated['alert'] = val_scores[name], val_scores[name] >= threshold['threshold']
            for (seed, env), group in evaluated.groupby(['seed', 'environment']):
                anomaly_rows.append(dict(fold=fold, seed=seed, environment=env, model=name, **anomaly_metrics(group.label, group.score, group.alert)))
    fs, ans = pd.DataFrame(forecast_rows), pd.DataFrame(anomaly_rows)
    # Equal weight to each seed/environment/fold; worst environment is averaged over seeds/folds.
    rank_f = fs.groupby('model').WAPE.mean().rename('mean_WAPE').to_frame()
    rank_f['worst_environment_WAPE'] = fs.groupby(['model', 'environment']).WAPE.mean().groupby('model').max()
    rank_f['selection_objective'] = (rank_f.mean_WAPE + rank_f.worst_environment_WAPE) / 2
    best_f = rank_f.sort_values(['selection_objective', 'mean_WAPE'], kind='stable').index[0]
    rank_a = ans.groupby('model')[['precision', 'recall', 'F1', 'FPR']].mean()
    rank_a['worst_environment_F1'] = ans.groupby(['model', 'environment']).F1.mean().groupby('model').min()
    rank_a['selection_objective'] = (rank_a.F1 + rank_a.worst_environment_F1) / 2
    # Threshold constraints apply on calibration; validation violations remain visible, not hidden.
    best_a = rank_a.sort_values(['selection_objective', 'precision'], ascending=False, kind='stable').index[0]
    print('Forecast selection (validation only):\n', rank_f.to_string())
    print('Anomaly selection (validation only):\n', rank_a.to_string())
    print('Fold threshold feasibility:\n', pd.DataFrame(threshold_rows).to_string(index=False))
    return best_f, best_a, fs, ans, pd.DataFrame(threshold_rows), rank_f, rank_a

def freeze_models(data, forecast_name, anomaly_name):
    f, fx, a, ax = (data[k] for k in ('f', 'fx', 'a', 'ax'))
    fit_end, cal_start, cal_end = int(.72 * CFG['weeks']), int(.78 * CFG['weeks']), int(.90 * CFG['weeks'])
    forecast_fit = f.cohort.eq('known') & (f.week + 1 <= cal_end)
    anomaly_fit = a.cohort.eq('known') & a.week.lt(fit_end)
    calibration = a.cohort.eq('known') & a.week.ge(cal_start) & a.week.lt(cal_end)
    anomaly_fit, calibration = purge_event_masks(a, [anomaly_fit, calibration])
    assert not (anomaly_fit & calibration).any()
    assert a.loc[anomaly_fit, 'time'].max() < a.loc[calibration, 'time'].min()
    for fold in (0, 1):
        _, _, selection = split_masks(a, fold)
        assert a.loc[selection, 'time'].max() < a.loc[calibration, 'time'].min()
    forecast_model = fit_forecast(forecast_name, fx.loc[forecast_fit], f.loc[forecast_fit])
    anomaly_model = fit_anomaly(ax.loc[anomaly_fit])
    scores = score_anomaly(anomaly_model, ax.loc[calibration])
    thresholds = {name: choose_threshold(a.loc[calibration, 'label'], scores[name]) for name in ANOMALY_CANDIDATES}
    calibration_results = {name: anomaly_metrics(a.loc[calibration, 'label'], scores[name],
                           scores[name] >= thresholds[name]['threshold']) for name in ANOMALY_CANDIDATES}
    print('Frozen selection:', forecast_name, anomaly_name)
    print('All later-calibration thresholds:', thresholds)
    return dict(forecast_name=forecast_name, anomaly_name=anomaly_name, forecast_model=forecast_model,
                anomaly_model=anomaly_model, threshold=thresholds[anomaly_name], thresholds=thresholds,
                calibration_metrics=calibration_results, calibration_rows=int(calibration.sum()),
                fit_end=fit_end, cal_start=cal_start, cal_end=cal_end,
                forecast_features=F_FEATURES, anomaly_features=A_FEATURES)

best_f, best_a, validation_forecast, validation_anomaly, fold_thresholds, forecast_ranking, anomaly_ranking = select_models(development)
frozen = freeze_models(development, best_f, best_a)


In [ ]:
def cluster_intervals(frame, kind):
    rng = np.random.default_rng(883)
    if kind == 'forecast':
        work = frame.assign(error=np.abs(frame.prediction - frame.y))
        sums = work.groupby('user')[['error', 'y']].sum().to_numpy()
        names = ['WAPE']
    else:
        y, alert = frame.label.astype(bool), frame.alert.astype(bool)
        work = frame.assign(tp=(y & alert).astype(int), fp=(~y & alert).astype(int), fn=(y & ~alert).astype(int))
        sums = work.groupby('user')[['tp', 'fp', 'fn']].sum().to_numpy()
        names = ['precision', 'recall', 'F1']
    estimates = []
    for _ in range(CFG['bootstrap']):
        sampled = sums[rng.integers(len(sums), size=len(sums))].sum(axis=0)
        if kind == 'forecast':
            estimates.append([sampled[0] / sampled[1] if sampled[1] else np.nan])
        else:
            tp, fp, fn = sampled
            p, r = tp / max(1, tp + fp), tp / max(1, tp + fn)
            estimates.append([p, r, 2 * p * r / max(1e-15, p + r)])
    bounds = np.nanquantile(estimates, [.025, .975], axis=0)
    return {f'{name}_{side}': float(bounds[j, i]) for i, name in enumerate(names) for j, side in enumerate(('lo', 'hi'))}

def event_results(evaluated, all_anomalies):
    onset = all_anomalies.loc[all_anomalies.label.eq(1)].groupby('event').time.min()
    events = []
    for (model, event), group in evaluated.loc[evaluated.label.eq(1)].groupby(['model', 'event']):
        alerts = group.loc[group.alert]
        first = onset.loc[event]
        events.append(dict(model=model, selected=bool(group.selected.iloc[0]), event=event, family=group.family.iloc[0], user=group.user.iloc[0],
                           seed=group.seed.iloc[0], environment=group.environment.iloc[0],
                           onset=first, left_censored=bool(group.time.min() > first), affected_transactions=len(group),
                           detected=bool(len(alerts)), delay_hours=(alerts.time.min() - first).total_seconds() / 3600 if len(alerts) else np.nan))
    return pd.DataFrame(events, columns=['model', 'selected', 'event', 'family', 'user', 'seed', 'environment', 'onset',
                                        'left_censored', 'affected_transactions', 'detected', 'delay_hours'])

def evaluate(data, fitted, split, final=False):
    f, fx, a, ax = (data[k] for k in ('f', 'fx', 'a', 'ax'))
    if final:
        fm, am = f.week.ge(8), a.week.ge(8)
    elif split == 'development_unseen':
        fm, am = f.cohort.eq('unseen'), a.cohort.eq('unseen') & a.week.ge(8)
    else:
        fm = f.cohort.eq('known') & f.week.ge(fitted['cal_end'])
        am = a.cohort.eq('known') & a.week.ge(fitted['cal_end'])
        assert f.loc[f.cohort.eq('known') & (f.week + 1 <= fitted['cal_end']), 'target_end'].max() <= f.loc[fm, 'time'].min()
    fp, ap = f.loc[fm].copy(), a.loc[am].copy()
    fp['prediction'] = predict_forecast(fitted['forecast_name'], fitted['forecast_model'], fx.loc[fm], fp)
    fp['model'], fp['selected'] = fitted['forecast_name'], True
    scores = score_anomaly(fitted['anomaly_model'], ax.loc[am])
    ap = pd.concat([ap.assign(model=name, selected=name == fitted['anomaly_name'], score=scores[name],
                              alert=scores[name] >= fitted['thresholds'][name]['threshold'])
                    for name in ANOMALY_CANDIDATES], ignore_index=True)
    fp['split'], ap['split'] = split, split
    forecast_rows, anomaly_rows, family_rows = [], [], []
    for (seed, env), group in fp.groupby(['seed', 'environment']):
        for model in dict.fromkeys([fitted['forecast_name'], 'last', 'mean4', 'recurring']):
            pred = group.prediction if model == fitted['forecast_name'] else group[model]
            row = dict(split=split, seed=seed, environment=env, model=model, selected=model == fitted['forecast_name'], **forecast_metrics(group.y, pred))
            if model == fitted['forecast_name']:
                row.update(cluster_intervals(group, 'forecast'))
            forecast_rows.append(row)
    events = event_results(ap, a)
    events['split'] = split
    for (seed, env, model), group in ap.groupby(['seed', 'environment', 'model']):
        ev = events.loc[events.seed.eq(seed) & events.environment.eq(env) & events.model.eq(model)]
        eligible = ev.loc[~ev.left_censored.astype(bool)]
        row = dict(split=split, seed=seed, environment=env, model=model, selected=model == fitted['anomaly_name'],
                   **anomaly_metrics(group.label, group.score, group.alert), **cluster_intervals(group, 'anomaly'))
        row['evaluation_constraints_met'] = bool(row['TP'] + row['FP'] > 0 and row['precision'] >= MIN_PRECISION and row['FPR'] <= MAX_FPR)
        row.update(event_support=len(eligible), left_censored_events=len(ev) - len(eligible),
                   event_recall=float(eligible.detected.mean()) if len(eligible) else np.nan,
                   detected_event_median_delay_hours=float(eligible.delay_hours.median()) if eligible.delay_hours.notna().any() else np.nan)
        anomaly_rows.append(row)
        for family in ('large', 'burst_duplicate', 'recurring_change', 'gradual', 'split'):
            family_ev = eligible.loc[eligible.family.eq(family)]
            affected = group.loc[group.family.eq(family)]
            family_rows.append(dict(split=split, seed=seed, environment=env, model=model, selected=model == fitted['anomaly_name'], family=family,
                                    transaction_support=len(affected), transaction_recall=float(affected.alert.mean()) if len(affected) else np.nan,
                                    event_support=len(family_ev), event_recall=float(family_ev.detected.mean()) if len(family_ev) else np.nan,
                                    median_delay_hours=float(family_ev.delay_hours.median()) if family_ev.delay_hours.notna().any() else np.nan))
    return dict(forecast_metrics=pd.DataFrame(forecast_rows), anomaly_metrics=pd.DataFrame(anomaly_rows),
                family_metrics=pd.DataFrame(family_rows), forecast_predictions=fp, anomaly_predictions=ap, events=events)

def combine_results(results):
    return {key: pd.concat([result[key] for result in results], ignore_index=True) for key in results[0]}

def show_summary(results):
    for kind, columns in [('forecast', ['WAPE', 'MAE', 'bias']), ('anomaly', ['precision', 'recall', 'F1', 'FPR', 'event_recall'])]:
        metrics = results[kind + '_metrics']
        summary = metrics.groupby(['split', 'environment', 'model'])[columns].agg(['mean', 'std', 'min', 'max', 'count', 'size'])
        print(kind, 'all-seed summaries (no seed selection):\n', summary.round(4).to_string())
    print('Per-seed anomaly support and alerts:\n', results['anomaly_metrics'][['split', 'seed', 'environment', 'model', 'selected', 'positive_support', 'negative_support', 'TP', 'FP', 'FN', 'event_support']].to_string(index=False))


In [ ]:
if (OUTPUT_DIR / 'final_consumed.json').exists():
    raise RuntimeError('Consumed final: export/refreeze writes are forbidden, including cell reruns.')
_FINAL_WRITE_TOKEN = None

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer, np.bool_)):
        return value.item()
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    return value

def assert_writable(token=None):
    if (OUTPUT_DIR / 'final_consumed.json').exists() and (token is None or token is not _FINAL_WRITE_TOKEN):
        raise RuntimeError('Consumed final: writes forbidden outside the active one-shot final export.')

def export_tables(results, prefix, token=None):
    assert_writable(token)
    for name, frame in results.items():
        frame.to_csv(OUTPUT_DIR / f'{prefix}_{name}.csv', index=False)
    for kind in ('forecast', 'anomaly'):
        metrics = results[kind + '_metrics']
        columns = ['WAPE', 'MAE', 'RMSE', 'R2', 'bias'] if kind == 'forecast' else ['precision', 'recall', 'F1', 'FPR', 'AP', 'ROC_AUC', 'event_recall']
        metrics.groupby(['split', 'environment', 'model'])[columns].agg(['mean', 'std', 'min', 'max', 'count', 'size']).to_csv(OUTPUT_DIR / f'{prefix}_{kind}_seed_summary.csv')

def runtime_state():
    return json_safe(dict(mode=MODE, modes=MODES, config=CFG, development_seeds=DEV_SEEDS, final_seeds=FINAL_SEEDS,
                          environments=ENVIRONMENTS, forecast_features=F_FEATURES, anomaly_features=A_FEATURES,
                          forecast_candidates=FORECAST_CANDIDATES, anomaly_candidates=ANOMALY_CANDIDATES,
                          min_precision=MIN_PRECISION, max_FPR=MAX_FPR, output_dir=str(OUTPUT_DIR.resolve()),
                          selected_forecast=best_f, selected_anomaly=best_a, versions=VERSIONS,
                          actual_versions={dist: importlib.metadata.version(dist) for dist in PACKAGES.values()}))

def implementation_fingerprints():
    names = ('generate complete_mondays recurrence build_forecast build_anomaly prepare forecast_metrics '
             'anomaly_metrics choose_threshold fit_forecast predict_forecast rule_scores fit_anomaly score_anomaly '
             'split_masks purge_event_masks run_correctness_tests select_models freeze_models cluster_intervals '
             'event_results evaluate combine_results show_summary json_safe assert_writable export_tables '
             'runtime_state implementation_fingerprints verify_snapshot run_final').split()
    # Format 2 avoids reference/intern-table changes after execution on CPython 3.13.
    # Nested function/lambda code and literal scoring/generator constants are included.
    return {name: hashlib.sha256(marshal.dumps(globals()[name].__code__, 2) +
                 repr((globals()[name].__defaults__, globals()[name].__kwdefaults__)).encode()).hexdigest() for name in names}

def verify_snapshot():
    assert_writable()
    snapshot_bytes = (OUTPUT_DIR / 'freeze_snapshot.json').read_bytes()
    if hashlib.sha256(snapshot_bytes).hexdigest() != snapshot_file_sha256:
        raise RuntimeError('Freeze snapshot changed')
    snapshot = json.loads(snapshot_bytes)
    manifest_bytes = (OUTPUT_DIR / 'frozen_manifest.json').read_bytes()
    model_bytes = (OUTPUT_DIR / 'frozen_models.joblib').read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != snapshot['manifest_sha256']:
        raise RuntimeError('Frozen manifest file changed')
    if hashlib.sha256(model_bytes).hexdigest() != snapshot['model_sha256']:
        raise RuntimeError('Frozen model file changed')
    saved = json.loads(manifest_bytes)
    if saved != json_safe(manifest) or saved['runtime_state'] != runtime_state():
        raise RuntimeError('Manifest/config/seeds/environments/features changed after freeze')
    fingerprints = implementation_fingerprints()
    if saved['implementation'] != fingerprints:
        raise RuntimeError('Live implementation changed after freeze')
    digest = hashlib.sha256(model_bytes + manifest_bytes + json.dumps(fingerprints, sort_keys=True).encode()).hexdigest()
    if digest != snapshot['freeze_sha256'] or digest != freeze_hash:
        raise RuntimeError('Combined provenance hash changed')
    # Deserialize only the exact OWN bytes just verified, not a mutable in-memory estimator or reread path.
    return joblib.load(io.BytesIO(model_bytes)), snapshot

def run_final():
    global _FINAL_WRITE_TOKEN
    verified, snapshot = verify_snapshot()
    with (OUTPUT_DIR / 'final_consumed.json').open('x', encoding='utf-8') as handle:
        json.dump(dict(seeds=FINAL_SEEDS, environments=ENVIRONMENTS, **snapshot,
                       consumed_at=datetime.now(timezone.utc).isoformat(), status='consumed before evaluation'), handle, indent=2)
    _FINAL_WRITE_TOKEN = object()
    try:
        final_data = prepare(FINAL_SEEDS, final=True)
        assert set(final_data['profiles'].user).isdisjoint(development['profiles'].user)
        results = evaluate(final_data, verified, 'final_new_users_process', final=True)
        expected = {(s, e, m) for s in FINAL_SEEDS for e in ENVIRONMENTS for m in ANOMALY_CANDIDATES}
        assert set(results['anomaly_metrics'][['seed', 'environment', 'model']].itertuples(index=False, name=None)) == expected
        export_tables(results, 'final', token=_FINAL_WRITE_TOKEN)
        show_summary(results)
        return results
    finally:
        _FINAL_WRITE_TOKEN = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = dict(protocol='spendly-research-v3', currency='KES', median_monthly_income=34000, mode=MODE, config=CFG, development_seeds=DEV_SEEDS,
                final_seeds=FINAL_SEEDS, environments=ENVIRONMENTS, versions=VERSIONS,
                forecast_features=F_FEATURES, anomaly_features=A_FEATURES,
                forecast_candidates=FORECAST_CANDIDATES, anomaly_candidates=ANOMALY_CANDIDATES,
                selected_forecast=best_f, selected_anomaly=best_a,
                forecast_model_params=frozen['forecast_model'].get_params(deep=False) if frozen['forecast_model'] is not None else {},
                isolation_forest_params=frozen['anomaly_model']['model'].get_params(),
                anomaly_score_config=dict(reference_max_rows=20000, reference_seed=47, hybrid_rules_weight=.65,
                                          hybrid_iforest_weight=.35, cdf_side='right',
                                          rules_weights=dict(log_category_deviation=1., log_amount_ratio=.6,
                                                             log_merchant_ratio=.7, log_duplicate_hour=.9,
                                                             log_count_hour=.3, night_rarity=.2)),
                rolling_folds=[dict(train_end=int(CFG['weeks'] * fraction),
                                    calibration_weeks=max(1, int(.08 * CFG['weeks'])),
                                    validation_weeks=max(2, int(.10 * CFG['weeks']))) for fraction in (.40, .52)],
                constraints=dict(min_precision=MIN_PRECISION, max_FPR=MAX_FPR, fallback='no_alert_above_1'),
                threshold=frozen['threshold'], thresholds=frozen['thresholds'], calibration_metrics=frozen['calibration_metrics'],
                calibration_rows=frozen['calibration_rows'], event_purge='full event containment per selection mask; feature streams preserved',
                split_weeks=dict(anomaly_fit_end=frozen['fit_end'], calibration_start=frozen['cal_start'],
                                 calibration_end=frozen['cal_end'], forecast_fit_target_available_through=frozen['cal_end']),
                selection=dict(forecast='mean and worst environment validation WAPE, equally weighted',
                               anomaly='mean and worst environment validation F1, equally weighted'),
                limitations=['synthetic only', 'point-estimate constraints', 'conditional user bootstrap',
                             'no supervised classifier', 'no real-data performance claim'])
# Pipeline objects are represented by explicit estimator parameters rather than repr-only blobs.
if best_f == 'ridge_ratio':
    manifest['forecast_model_params'] = {name: step.get_params() for name, step in frozen['forecast_model'].steps}
manifest.update(runtime_state=runtime_state(), implementation=implementation_fingerprints())
frozen_json = json.dumps(json_safe(manifest), indent=2, sort_keys=True, allow_nan=False)
manifest_bytes = frozen_json.encode('utf-8')
(OUTPUT_DIR / 'frozen_manifest.json').write_bytes(manifest_bytes)
joblib.dump(frozen, OUTPUT_DIR / 'frozen_models.joblib')
model_bytes = (OUTPUT_DIR / 'frozen_models.joblib').read_bytes()
freeze_hash = hashlib.sha256(model_bytes + manifest_bytes + json.dumps(manifest['implementation'], sort_keys=True).encode()).hexdigest()
snapshot = dict(freeze_sha256=freeze_hash, model_sha256=hashlib.sha256(model_bytes).hexdigest(),
                manifest_sha256=hashlib.sha256(manifest_bytes).hexdigest())
snapshot_bytes = json.dumps(snapshot, indent=2, sort_keys=True).encode()
snapshot_file_sha256 = hashlib.sha256(snapshot_bytes).hexdigest()
(OUTPUT_DIR / 'freeze_snapshot.json').write_bytes(snapshot_bytes)
verify_snapshot()
for name, frame in [('validation_forecast', validation_forecast), ('validation_anomaly', validation_anomaly),
                    ('fold_thresholds', fold_thresholds), ('forecast_ranking', forecast_ranking.reset_index()),
                    ('anomaly_ranking', anomaly_ranking.reset_index())]:
    frame.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
development_results = combine_results([evaluate(development, frozen, 'known_user_future'),
                                       evaluate(development, frozen, 'development_unseen')])
export_tables(development_results, 'development')
show_summary(development_results)
print('Combined model/manifest/implementation SHA256:', freeze_hash)
print('Outputs:', OUTPUT_DIR.resolve(), '| elapsed seconds:', round(time.perf_counter() - STARTED, 1))


## Final Gate
Default Run all stops at development evaluation. For a deliberate final run, set `SPENDLY_RUN_FINAL=1` before configuration (or explicitly set `RUN_FINAL = True` here after freezing). Do not opt in while debugging or choosing features, candidates, thresholds, or seeds. Final verifies model bytes, manifest, runtime config/seeds/features, and marshaled live implementation fingerprints including dependent helpers and literal constants. It loads only the verified OWN joblib bytes, ignoring mutable in-memory frozen models. All declared anomaly candidates run without reselection. The marker is written BEFORE generating final data; a failed attempt is conservatively consumed too. Only that active one-shot final writer may finish result exports after the marker; subsequent result export/refreeze writes are rejected. The optional ZIP cell may still archive these existing results without modifying them. Hashes detect accidental drift, not a malicious actor able to rewrite this runtime and its files.

Changing mode or directory does not make previously viewed seeds fresh. After final results, use genuinely new precommitted seeds for subsequent research, and treat these results as development knowledge. Exported joblib models are for trusted local research only: loading an untrusted pickle/joblib file can execute code. CSV predictions include labels for auditing, but label columns never enter a feature matrix.


In [ ]:
final_results = None
assert_writable()
# Plot before the final marker, so a consumed directory is never rewritten on rerun.
plot_data = development_results['forecast_metrics']
selected = plot_data.loc[plot_data.model.eq(best_f)]
fig, ax = plt.subplots(figsize=(8, 3))
for split, group in selected.groupby('split'):
    ax.plot([f'{e}/{s}' for e, s in zip(group.environment, group.seed)], group.WAPE, 'o', label=split)
ax.set(ylabel='WAPE (lower is better)', title='Development evaluation only: selected forecast, every seed')
ax.legend(); fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'development_wape.png', dpi=120)
plt.show()
plt.close(fig)
if RUN_FINAL:
    final_results = run_final()
else:
    print('Final not generated or evaluated. Set SPENDLY_RUN_FINAL=1 only after freezing; viewed seeds are consumed.')

print('Research run complete in', round(time.perf_counter() - STARTED, 1), 'seconds. No real-world accuracy claim.')


## Optional Export
This separate guarded cell does nothing during default Run all. Set `SPENDLY_EXPORT_ZIP=1` to create a local archive; additionally set `SPENDLY_COLAB_DOWNLOAD=1` to explicitly request Colab download. Archiving is allowed after final evaluation: it reads existing results and writes the ZIP outside the protected results directory. Seed summaries include count (defined values) and size (total seed values). No application artifacts are touched. Quick is the default, full is larger, smoke is for correctness rather than model quality.

Local smoke execution needs only the installed packages above. Set `SPENDLY_MODE=smoke`, `SPENDLY_RUN_FINAL=0`, `MPLBACKEND=Agg`, and optionally `SPENDLY_OUTPUT_DIR` to a disposable research directory. Read the notebook with `json`, parse each code cell with `ast.parse`, and `exec(compile(...), namespace)` in order using the same namespace. There are no IPython magics, notebook-only display calls, or repository imports.


In [ ]:
if os.environ.get('SPENDLY_EXPORT_ZIP', '0') == '1':
    import zipfile
    archive = OUTPUT_DIR.parent / (OUTPUT_DIR.name + '.zip')
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
        for path in sorted(OUTPUT_DIR.rglob('*')):
            if path.is_file():
                zipped.write(path, arcname=str(path.relative_to(OUTPUT_DIR)))
    print('Research archive:', archive.resolve())
    if os.environ.get('SPENDLY_COLAB_DOWNLOAD', '0') == '1':
        try:
            from google.colab import files
        except ImportError:
            print('Not in Colab; archive remains local.')
        else:
            files.download(str(archive))
